# Chapter 11: Context Engineering, the Three-Tier Knowledge Index (Personal Finance Example)

This notebook shows the central problem of Chapter 11 with a personal-finance example, using pure LLM calls. No coding assistant, no hidden retrieval.

Every time you open a new session, an AI assistant starts as a junior. It knows nothing about your situation. You either re-paste everything, which burns tokens, time, and context window, or you give it a small index it can navigate to load only what it needs.

The knowledge folder is organized in three tiers:

- T0, `AGENT.md`: the only file read every session. Hard rules, precedence order, and how to navigate.
- T1, `INDEX.md`: generated, one line per item, grouped by directory. This is the map.
- T2, content: the files themselves, loaded only when the index points at them.

The directory name is the type: `rules/ entities/ facts/ decisions/ archive/`, and that order is the precedence order. The persona uses my own name, Rany ElHousieny, but every number is invented. The artifacts live in the `finance_agent` folder next to this notebook.

The measurable parts (context assembly and token counts) run with no API key. The live model comparison runs if `OPENAI_API_KEY` is set, and skips politely if not. Run the cells top to bottom.

In [1]:
# === Setup ===
# A context layer is files and discipline, not another platform.
import os, sys, io, re, shutil, subprocess, tempfile, contextlib
from pathlib import Path

# Locate the illustrative finance_agent artifacts (Rany ElHousieny; all numbers invented).
candidates = [Path('../finance_agent'), Path('finance_agent'), Path('chapter_11/finance_agent')]
AGENT = next((c for c in candidates if c.exists()), None)
assert AGENT is not None, 'Could not find the finance_agent folder. Run from the chapter_11 area of the repo.'

# Pin a reference "as of" date so the staleness advisories in Part 7 are reproducible:
# you get the same numbers the book shows. In production the validator falls back to the
# real clock, because a real repo must report true staleness. This only freezes the demo.
os.environ.setdefault('FINANCE_AGENT_TODAY', '2026-07-25')

def tokens(text):
    # Rough token estimate: about 4 characters per token for English text.
    return len(text) // 4

def read(rel):
    return (AGENT / rel).read_text()

# Model selection is a book-wide rule: never hardcode a model. Import the shared
# helper that discovers, tests, and self-heals. Fall back to an inline copy so the
# notebook still runs on Colab if utils is not on the path.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'utils' / 'notebook_setup.py').exists():
        sys.path.insert(0, str(_p)); break
try:
    from utils.notebook_setup import select_and_test_model
except Exception:
    def select_and_test_model(client, preferred=('gpt-4o', 'gpt-4.1', 'gpt-4o-mini', 'gpt-4-turbo', 'gpt-4', 'gpt-3.5-turbo')):
        available = {m.id for m in client.models.list()}
        ordered = [m for m in preferred if m in available]
        ordered += sorted((m for m in available if m.startswith('gpt-') and m not in ordered), reverse=True)
        for name in ordered:
            try:
                client.chat.completions.create(model=name, messages=[{'role': 'user', 'content': 'ping'}], max_tokens=1)
                print('Selected and tested model:', name); return name
            except Exception as err:
                print('Skipping', name, '(' + type(err).__name__ + ')')
        raise RuntimeError('No usable chat model found for this API key.')

# No hardcoded model. Discover and verify one at runtime when a key is present.
# Only prompt in an interactive session; skip cleanly under automated execution.
if not os.environ.get('OPENAI_API_KEY') and sys.stdin and sys.stdin.isatty():
    try:
        import getpass
        entered = getpass.getpass('Enter your OPENAI_API_KEY (leave blank to skip live steps): ').strip()
        if entered:
            os.environ['OPENAI_API_KEY'] = entered
    except Exception:
        pass

client = None
MODEL = None
if os.environ.get('OPENAI_API_KEY'):
    from openai import OpenAI
    client = OpenAI()
    with contextlib.redirect_stdout(io.StringIO()):
        MODEL = select_and_test_model(client)

# Confirm the knowledge folder resolved, split into the three tiers.
all_md = sorted(str(p.relative_to(AGENT)) for p in AGENT.rglob('*.md'))
content_md = [f for f in all_md if f not in ('AGENT.md', 'INDEX.md')]
print('Found the knowledge folder.')
print('  T0 contract (AGENT.md):', 'AGENT.md' in all_md)
print('  T1 index (INDEX.md)   :', 'INDEX.md' in all_md)
print('  T2 content            :', len(content_md), 'files')
for f in content_md:
    print('     ', f)

Found the knowledge folder.
  T0 contract (AGENT.md): True
  T1 index (INDEX.md)   : True
  T2 content            : 8 files
      archive/2025_allocation.md
      decisions/2026-01-10_annual_review.md
      decisions/2026-03-14_529_vs_brokerage.md
      entities/advisor_okonkwo.md
      entities/employer_equity.md
      facts/accounts.md
      facts/allocation_2026.md
      rules/decision_rules.md


## Part 1: The junior problem (no context)

A fresh session. We ask about the finances with no context at all. This is exactly what a new chat sees. Watch the model guess or refuse. That is the cost of starting junior every single time.

In [2]:
# === Part 1: The junior problem ===
QUESTION = 'What is my current target allocation, and can I add more to my tech stock?'

def build_prompt(context, question):
    if context:
        return 'Context:\n' + context + '\n\nQuestion: ' + question
    return 'Question: ' + question

def call_llm(context, question):
    # Pure LLM call. No tools, no retrieval framework, no coding assistant.
    if client is None:
        return None
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer only from the provided context. If the context is empty or does not contain the answer, say you do not have that information. Cite the file you used.'},
            {'role': 'user', 'content': build_prompt(context, question)},
        ],
        temperature=0.0,
    )
    return resp.choices[0].message.content

print('Question:', QUESTION)
print('Context sent: 0 tokens (the model knows nothing about the persona)')
answer = call_llm('', QUESTION)
if answer is None:
    print()
    print('[No OPENAI_API_KEY set. With no context, a fresh model can only guess or refuse.]')
else:
    print()
    print('Junior answer (no context):')
    print(answer)

Question: What is my current target allocation, and can I add more to my tech stock?
Context sent: 0 tokens (the model knows nothing about the persona)

[No OPENAI_API_KEY set. With no context, a fresh model can only guess or refuse.]


## Part 2: The brute-force fix (dump everything)

The lazy fix is to paste every file into the prompt and hope. It often works, but watch two things: the token count, and the fact that the dump carries the superseded 2025 allocation right next to the current one, with no signal about which is true.

In [3]:
# === Part 2: The context dump (naive assembly) ===
def dump_everything():
    parts = []
    for rel in content_md:
        parts.append('----- ' + rel + ' -----')
        parts.append(read(rel))
    return '\n'.join(parts)

blob = dump_everything()
print('Context dump:', tokens(blob), 'tokens from', len(content_md), 'content files')
print()
print('Order the model sees the files in:')
for i, f in enumerate(content_md, 1):
    print('  ', i, f)
print()
print('The dump includes the SUPERSEDED 2025 allocation next to the current one,')
print('with nothing telling the model which is true today:')
print('  archive/2025_allocation.md : 80/20 (last year, superseded)')
print('  facts/allocation_2026.md   : 60/40 (current)')
print()
dump_answer = call_llm(blob, QUESTION)
if dump_answer is None:
    print('[No API key. The dump would answer, but at', tokens(blob), 'tokens with the stale file in the window.]')
else:
    print('Answer from the full dump:')
    print(dump_answer)

Context dump: 1156 tokens from 8 content files

Order the model sees the files in:
   1 archive/2025_allocation.md
   2 decisions/2026-01-10_annual_review.md
   3 decisions/2026-03-14_529_vs_brokerage.md
   4 entities/advisor_okonkwo.md
   5 entities/employer_equity.md
   6 facts/accounts.md
   7 facts/allocation_2026.md
   8 rules/decision_rules.md

The dump includes the SUPERSEDED 2025 allocation next to the current one,
with nothing telling the model which is true today:
  archive/2025_allocation.md : 80/20 (last year, superseded)
  facts/allocation_2026.md   : 60/40 (current)

[No API key. The dump would answer, but at 1156 tokens with the stale file in the window.]


## Part 3: The Knowledge Map way (T0 plus a generated index)

Instead of dumping, we read the small map and open only what it points at. Two changes from the old design. First, the map is generated from each file's own frontmatter, so it cannot drift out of sync with the files by hand. We run the generator live below and watch the index appear. Second, the map is plain markdown links, so it parses with one regex and no YAML library. The parser is shorter than the YAML version it replaces, which is the point.

In [4]:
# === Part 3: Map-driven assembly ===
# Generate the index live, from each file's own frontmatter. Watch it appear.
gen = subprocess.run([sys.executable, str(AGENT / 'tools' / 'build_index.py')], capture_output=True, text=True)
print(gen.stdout.strip())
print()
print('--- INDEX.md (T1, generated) ---')
print(read('INDEX.md'))

# Parse the index with ONE regex. No YAML import. This is the whole parser.
LINK = re.compile(r'\[([^\]]+)\]\(([^)]+\.md)\)')

def parse_index():
    entries, section = [], None
    for line in read('INDEX.md').splitlines():
        s = line.strip()
        if s.startswith('## '):
            section = s[3:].strip()
        m = LINK.search(s)
        if m:
            path = m.group(2)
            hook = s.split(') - ', 1)[1] if ') - ' in s else ''
            entries.append({'label': m.group(1), 'path': path, 'hook': hook,
                            'section': section, 'dir': path.split('/')[0]})
    return entries

# Directory name is the type; this order is the precedence order.
PREC = {'rules': 0, 'entities': 1, 'facts': 2, 'decisions': 3, 'archive': 4}
STOP = {'my', 'the', 'a', 'an', 'is', 'are', 'do', 'does', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
        'what', 'how', 'i', 'me', 'can', 'should', 'with', 'it', 'that', 'this', 'more', 'add', 'current', 'some'}

def score_entries(entries, question):
    q = {w.strip('?.,').lower() for w in question.split()} - STOP
    out = []
    for e in entries:
        words = set(re.findall(r'[a-z0-9]+', (e['label'] + ' ' + e['hook']).lower()))
        out.append((sum(1 for w in q if w in words), e))
    return out

def pick(entries, question, allow_archive=False):
    chosen = [e for s, e in score_entries(entries, question) if s > 0 and (allow_archive or e['dir'] != 'archive')]
    for e in entries:                 # rules always apply, even when nothing matched
        if e['dir'] == 'rules' and e not in chosen:
            chosen.append(e)
    chosen.sort(key=lambda e: PREC.get(e['dir'], 9))
    return chosen

def assemble(entries):
    parts = []
    for e in entries:
        parts.append('----- ' + e['path'] + ' -----')
        parts.append(read(e['path']))
    return '\n'.join(parts)

entries = parse_index()
print()
print('Index has', len(entries), 'items. Scoring each against the question:')
for s, e in score_entries(entries, QUESTION):
    print('  score', s, '|', e['path'])
chosen = pick(entries, QUESTION)
print()
print('Chosen (archive excluded, rules always included):')
for e in chosen:
    print('  ', e['path'])
context = assemble(chosen)

# Two honest costs. Code navigates: only the chosen files reach the model.
# Model navigates: the model also reads T0 and the index to make the choice.
T0, T1 = read('AGENT.md'), read('INDEX.md')
floor = tokens(T0) + tokens(T1)
code_nav = tokens(context)
model_nav = floor + code_nav
print()
print('Token cost for this question:')
print('  dump (all content)   :', tokens(blob))
print('  map, code navigates  :', code_nav, '(only the chosen files)')
print('  map, model navigates :', model_nav, '(AGENT.md', tokens(T0), 'plus INDEX.md', tokens(T1), 'plus chosen', code_nav, ')')
map_answer = call_llm(context, QUESTION)
if map_answer is None:
    print()
    print('[No API key. The map context is', code_nav, 'tokens and excludes the stale 2025 file.]')
else:
    print()
    print('Answer from the map-assembled context:')
    print(map_answer)

# The same accounting across five questions, each built backwards from a real failure.
DEMOS = [
    ('Q1 current target allocation', 'What is my current target allocation?'),
    ('Q2 who is my advisor, what recommended', 'Who is my advisor and what have they recommended?'),
    ('Q3 raise cash for the 529', 'Should I sell some stock to fund the 529?'),
    ('Q4 why the 529 over a brokerage', 'Why the 529 rather than a taxable brokerage?'),
    ('Q5 checking balance (control)', 'What is my checking balance?'),
]
print()
print('Per-question token cost (model navigates) vs the full dump:')
print('  %-42s %6s %6s %6s' % ('query', 'dump', 'map', 'ratio'))
tot_dump = tot_map = 0
for label, q in DEMOS:
    m = floor + tokens(assemble(pick(entries, q)))
    tot_dump += tokens(blob); tot_map += m
    print('  %-42s %6d %6d %5.2fx' % (label, tokens(blob), m, tokens(blob) / m))
print('  %-42s %6d %6d %5.2fx' % ('TOTAL', tot_dump, tot_map, tot_dump / tot_map))

INDEX.md written: 8 items

--- INDEX.md (T1, generated) ---
# Knowledge Index

<!-- GENERATED by tools/build_index.py - do not hand-edit. -->

## Rules
- [Decision rules](rules/decision_rules.md) - Risk tolerance, rebalancing triggers, giving plan, and the equity blackout rule. Overrides all facts on conflict.

## Entities
- [M. Okonkwo (financial advisor)](entities/advisor_okonkwo.md) - Independent CFP, fee-only. Owns rebalancing proposals and the annual review. Does NOT have trading authority.
- [Employer equity position](entities/employer_equity.md) - RSU vesting schedule and the blackout window that gates all equity sale advice.

## Facts
- [Accounts and balances](facts/accounts.md) - Checking, savings, brokerage, retirement. Invented balances.
- [Target allocation (current)](facts/allocation_2026.md) - The 60/40 target adopted January 2026. Supersedes the 2025 80/20 target.

## Decisions
- [2026 annual review](decisions/2026-01-10_annual_review.md) - Why the allocation moved from 

## Part 4: Facts that expire

Two files answer "what is my allocation": the current 60/40 and the archived 80/20. A plain text search cannot tell them apart. The map can, two ways: directory precedence (`archive/` is history, never current truth) and each file's `last_verified` date.

In [5]:
# === Part 4: Temporal precedence (demo Q1) ===
Q1 = 'What is my current target allocation?'
print('Question:', Q1)
print()
print('Baseline: search the corpus for the numbers.')
hits = []
for rel in content_md:
    for i, ln in enumerate(read(rel).splitlines(), 1):
        if '60/40' in ln or '80/20' in ln:
            hits.append((rel, i))
files = sorted({h[0] for h in hits})
print('  text search "60/40" or "80/20" ->', len(hits), 'hits across', len(files), 'files:')
for f in files:
    print('     ', f)
print()
print('  Both 80/20 (archive) and 60/40 (current) appear, with no ranking signal.')
print('  A search-first agent can answer 80/20 and cite a real file. That is the failure.')
print()
print('The map way: precedence excludes archive, last_verified shows how fresh each file is.')
chosen1 = pick(entries, Q1)
for e in chosen1:
    fm = read(e['path']).split('---')[1] if read(e['path']).startswith('---') else ''
    lv = next((ln.split(':', 1)[1].strip() for ln in fm.splitlines() if ln.startswith('last_verified')), '?')
    print('  loaded:', e['path'], '(last_verified', lv, ')')
print('  archive/2025_allocation.md is NOT loaded: history, never current truth.')
ans1 = call_llm(assemble(chosen1), Q1)
if ans1 is None:
    print()
    print('[No API key. The loaded context contains only the current 60/40 file.]')
else:
    print()
    print('Answer:')
    print(ans1)

Question: What is my current target allocation?

Baseline: search the corpus for the numbers.
  text search "60/40" or "80/20" -> 8 hits across 5 files:
      archive/2025_allocation.md
      decisions/2026-01-10_annual_review.md
      entities/advisor_okonkwo.md
      facts/allocation_2026.md
      rules/decision_rules.md

  Both 80/20 (archive) and 60/40 (current) appear, with no ranking signal.
  A search-first agent can answer 80/20 and cite a real file. That is the failure.

The map way: precedence excludes archive, last_verified shows how fresh each file is.
  loaded: rules/decision_rules.md (last_verified 2026-07-01 )
  loaded: facts/allocation_2026.md (last_verified 2026-07-01 )
  loaded: decisions/2026-01-10_annual_review.md (last_verified 2026-01-10 )
  archive/2025_allocation.md is NOT loaded: history, never current truth.

[No API key. The loaded context contains only the current 60/40 file.]


## Part 5: Rules the agent must follow, and why safety rules are not lazy-loaded

Some knowledge must be present on every turn, not fetched when a keyword happens to match. The blackout window that forbids selling employer equity is the example. Ask a question phrased about the 529, and a score-based loader never opens the equity file. The fix is to keep the hard rule in `AGENT.md` (T0), which is read every session and names the file to check.

In [6]:
# === Part 5: T0 safety rules (demo Q3) ===
Q3 = 'Should I sell some stock to fund the 529?'
print('Question:', Q3)
print()
print('Score-based lazy loading picks:')
for s, e in score_entries(entries, Q3):
    if s > 0:
        print('  score', s, '|', e['path'])
picked = [e['path'] for e in pick(entries, Q3)]
print('  -> loaded:', picked)
print('  entities/employer_equity.md scored 0 and was not selected by the query.')
print()
print('  If the blackout lived only in that file, the agent would advise a sale')
print('  without knowing sales are currently prohibited. That is the unsafe answer.')
print()
print('The fix: the blackout is in AGENT.md (T0), read every session, and it names the file:')
for ln in read('AGENT.md').splitlines():
    if 'blackout' in ln.lower():
        print('   ', ln.strip())
print()
print('  Because the rule is resident, the agent opens the equity file before any equity advice:')
print('  --- entities/employer_equity.md ---')
print(read('entities/employer_equity.md').split('---', 2)[2].strip())
ctx3 = read('AGENT.md') + '\n' + assemble(pick(entries, Q3)) + '\n' + read('entities/employer_equity.md')
ans3 = call_llm(ctx3, Q3)
if ans3 is None:
    print()
    print('[No API key. With the blackout window in context, the model should refuse a sale during it.]')
else:
    print()
    print('Answer:')
    print(ans3)

Question: Should I sell some stock to fund the 529?

Score-based lazy loading picks:
  score 1 | decisions/2026-03-14_529_vs_brokerage.md
  -> loaded: ['rules/decision_rules.md', 'decisions/2026-03-14_529_vs_brokerage.md']
  entities/employer_equity.md scored 0 and was not selected by the query.

  If the blackout lived only in that file, the agent would advise a sale
  without knowing sales are currently prohibited. That is the unsafe answer.

The fix: the blackout is in AGENT.md (T0), read every session, and it names the file:
    1. **Blackout windows.** Never propose selling employer equity between quarter-end and the earnings

  Because the rule is resident, the agent opens the equity file before any equity advice:
  --- entities/employer_equity.md ---
# Employer Equity

- **Holding:** 1,200 vested RSUs, 400 unvested (vest 2027-01-15).
- **Concentration:** 18% of total portfolio - above the 10% single-position ceiling I hold myself to.
- **Current blackout window:** 2026-06-30 thr

## Part 6: The entities tier, "who is X and what have they recommended?"

This is the new tier, and the strongest demonstration in the notebook. No document in the old design was about the advisor; they appeared incidentally inside decisions and facts. An index of documents has nothing to point at. A per-entity file does.

In [7]:
# === Part 6: Per-entity files (demo Q2) ===
Q2 = 'Who is my advisor and what have they recommended?'
print('Question:', Q2)
print()
print('Baseline: the advisor is not the subject of any document. Mentions are scattered:')
scattered = [rel for rel in content_md if 'okonkwo' in read(rel).lower() or 'advisor' in read(rel).lower()]
for f in scattered:
    print('     ', f)
print('  The answer would have to be reconstructed from mentions across those files.')
print()
print('The entities tier gives the question a home. One hop from the index:')
chosen2 = pick(entries, Q2)
for e in chosen2:
    print('  loaded:', e['path'])
print()
print('  --- entities/advisor_okonkwo.md ---')
print(read('entities/advisor_okonkwo.md').split('---', 2)[2].strip())
ans2 = call_llm(assemble(chosen2), Q2)
if ans2 is None:
    print()
    print('[No API key. The advisor file carries role, authority limits, and recommendation history.]')
else:
    print()
    print('Answer:')
    print(ans2)

Question: Who is my advisor and what have they recommended?

Baseline: the advisor is not the subject of any document. Mentions are scattered:
      decisions/2026-01-10_annual_review.md
      entities/advisor_okonkwo.md
      facts/allocation_2026.md
  The answer would have to be reconstructed from mentions across those files.

The entities tier gives the question a home. One hop from the index:
  loaded: rules/decision_rules.md
  loaded: entities/advisor_okonkwo.md

  --- entities/advisor_okonkwo.md ---
# M. Okonkwo - Financial Advisor

**Role:** independent fee-only CFP. Engaged 2024. **No trading authority** - proposals only; the human executes.

**What they own:** the annual review, rebalancing proposals, tax-loss harvesting suggestions.

**What they have recommended**
| Date | Recommendation | Outcome |
|---|---|---|
| 2026-01-10 | Move 80/20 to 60/40 ahead of retirement horizon | Accepted - see `decisions/2026-01-10_annual_review.md` |
| 2026-03-14 | Fund education via 529 rathe

## Part 7: The generator and the validator

The index is generated, not hand-written, and a validator guards it. Here we run the validator on a clean copy (it passes), then simulate the drift that kills hand-maintained indexes: we delete a line from the index by hand. The validator catches it and exits non-zero. Regenerating fixes it. This is why generation is mandatory, not optional. We work on a throwaway copy so the real folder is never left broken.

In [8]:
# === Part 7: Generate then validate ===
def run_tool(tool, root):
    r = subprocess.run([sys.executable, str(Path(root) / 'tools' / tool)], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

tmp = Path(tempfile.mkdtemp(prefix='finance_agent_'))
shutil.copytree(AGENT, tmp, dirs_exist_ok=True)
print('Working on a throwaway copy:', tmp.name)
print()

rc, out = run_tool('build_index.py', tmp); print('build_index :', out)
rc, out = run_tool('validate.py', tmp);   print('validate    : rc =', rc); print(out)
print()

# Simulate hand-drift: delete one real pointer line from the generated index.
idx = tmp / 'INDEX.md'
lines = idx.read_text().splitlines()
dropped = next(l for l in lines if 'advisor_okonkwo.md' in l)
idx.write_text('\n'.join(l for l in lines if l != dropped) + '\n')
print('Hand-edited the index: removed the line for entities/advisor_okonkwo.md')
rc, out = run_tool('validate.py', tmp); print('validate    : rc =', rc, '(non-zero means the drift was caught)'); print(out)
print()

# Fix by regenerating. The generator is the single source of truth.
rc, out = run_tool('build_index.py', tmp); print('regenerate  :', out)
rc, out = run_tool('validate.py', tmp);   print('validate    : rc =', rc); print(out)
shutil.rmtree(tmp)

Working on a throwaway copy: finance_agent_3bwjr57j

build_index : INDEX.md written: 8 items


validate    : rc = 0
checked 8 files, 8 pointers
  ADVISORY stale 133d: entities/advisor_okonkwo.md
  ADVISORY stale 196d: decisions/2026-01-10_annual_review.md
  ADVISORY stale 133d: decisions/2026-03-14_529_vs_brokerage.md
PASS - all checks green

Hand-edited the index: removed the line for entities/advisor_okonkwo.md
validate    : rc = 1 (non-zero means the drift was caught)
checked 8 files, 7 pointers
  ADVISORY stale 133d: entities/advisor_okonkwo.md
  ADVISORY stale 196d: decisions/2026-01-10_annual_review.md
  ADVISORY stale 133d: decisions/2026-03-14_529_vs_brokerage.md

FAIL:
  [2] not indexed: entities/advisor_okonkwo.md



regenerate  : INDEX.md written: 8 items


validate    : rc = 0
checked 8 files, 8 pointers
  ADVISORY stale 133d: entities/advisor_okonkwo.md
  ADVISORY stale 196d: decisions/2026-01-10_annual_review.md
  ADVISORY stale 133d: decisions/2026-03-14_529_vs_brokerage.md
PASS - all checks green


## When the token argument actually starts to matter

At this scale the token win is small and honest: the always-resident floor (AGENT.md plus INDEX.md) is about 36% of a roughly 1,700-token corpus, far above the 5% you want on a real system, because at ten files the floor is mostly irreducible rules. Below about a hundred documents, the index is not a performance optimization. It is what stops the agent being confidently wrong. The token savings only becomes the headline at a scale a book example cannot hold (Table 11.1 in the chapter):

| Corpus | Flat index size | Load every session? | What to do |
|---|---:|---|---|
| 10 files (this chapter) | ~350 tokens | yes, trivially | Read it all if you like. The index earns its place on correctness, not tokens. |
| ~100 files | ~3,000 tokens | yes | The index now saves tokens too. |
| ~400 files | ~10,000 tokens | yes, at its ceiling | A flat index is near its practical limit. |
| ~1,700 files (measured) | ~46,000 tokens | no, too big | Shard it: a ~600-token root index of areas, open one branch on demand. |

## Where to go from here

Swap the persona's folder for your own: your notes, your rules, your progress tracker. The three tiers do not change with the domain. Start with `AGENT.md`, generate `INDEX.md`, and keep three fields on every file: `name`, `description`, `last_verified`. The production version of this pattern is the open-source Agentic Repos framework: https://github.com/ranyelhousieny/Agentic-Repo